# `rfqvendor` — one row per bidder submission against an RFQ

Unity Catalog: `ingestion_framework_test.bid_data_exploration.rfqvendor`

Expected: the bidder-level record (maps to AGPOWER / AL Geemi / etc. in the sample data). Look for: bidder name/vendor id, round/revision number, submission date, total price if rolled up here, and status (submitted / disqualified / awarded).

## Run 1 — initial exploration ✅ *(run in Databricks — awaiting results to document findings)*

In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.rfqvendor

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM ingestion_framework_test.bid_data_exploration.rfqvendor

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor LIMIT 20

## Run 2 — follow-up queries ⏳ *(pending — not yet run)*

First pass findings (full write-up in `databricks/FINDINGS.md`):
- **412,896 rows** vs. `rfq`'s 30,338 — ~13.6 invited-vendor rows per RFQ on average. This is invitees, not just actual submitters.
- **`VENDOR` is a numeric code (`001052`), not a company name.** There's no vendor/company master table in our current 7 — need to find one to resolve codes to names like "AGPOWER"/"AL Geemi".
- **`RFQVENDORID` is this table's PK.** Almost certainly the FK `quotationline` points back with — pending `quotationline`'s `DESCRIBE` to confirm the exact column name (case matters for the underscore: is it `RFQVENDORID` or `RFQVENDOR_ID`?).
- **Sample `RFQNUM` values have a `-R1` suffix** (`D1752-R1`) — this may mean negotiation rounds are separate RFQ *records* (a new RFQNUM per round), not a round column on one RFQ. That would conflict with the `DISCOUNT_REVISION` theory from `rfq` — there may be two different "round" concepts (formal re-tender vs. post-bid discount loop). Needs clarifying with TAQA.
- **7 award/discount-cost columns are typed `binary`, not decimal** (`TOTALAWARDCOST`, `TOTALDISCOUNT`, `TOTBIDCOSTWDIS`, etc.) and every sample row shows the *same* value (`CGTqea+foUw=`) for all of them — looks encrypted/masked, not just null. Two proper decimal columns exist alongside them (`TOTALAWARDCOSTWDIS`, `TOTALAWARDCOSTWITHTAXWDIS`) but are null in this (2003-era) sample — worth checking if those are populated on recent rows instead.

### Does `RFQNUM` really carry a round suffix?

In [ ]:
%sql
SELECT regexp_extract(RFQNUM, '-(R[0-9]+)$', 1) AS round_suffix, COUNT(*) AS n
FROM ingestion_framework_test.bid_data_exploration.rfqvendor
GROUP BY round_suffix
ORDER BY n DESC

### Bid status distribution

In [ ]:
%sql
SELECT BIDSTATUS, COUNT(*) AS n FROM ingestion_framework_test.bid_data_exploration.rfqvendor
GROUP BY BIDSTATUS ORDER BY n DESC

### Is `DISCOUNT_REVISION` / `POSTBID_DISCOUNT_COUNTER` populated at the vendor level?

In [ ]:
%sql
SELECT DISCOUNT_REVISION, POSTBID_DISCOUNT_COUNTER, COUNT(*) AS n
FROM ingestion_framework_test.bid_data_exploration.rfqvendor
GROUP BY DISCOUNT_REVISION, POSTBID_DISCOUNT_COUNTER
ORDER BY n DESC
LIMIT 30

### Are the proper-decimal award-cost columns populated on recent rows?

In [ ]:
%sql
SELECT RFQNUM, VENDOR, ENTERDATE, TOTALAWARDCOSTWDIS, TOTALAWARDCOSTWITHTAXWDIS, SCORE, BIDSTATUS
FROM ingestion_framework_test.bid_data_exploration.rfqvendor
WHERE ENTERDATE >= '2024-01-01' AND TOTALAWARDCOSTWDIS IS NOT NULL
ORDER BY ENTERDATE DESC
LIMIT 30

### Recent rfqvendor rows for a sanity check (any resembling D-111808?)

In [ ]:
%sql
SELECT RFQNUM, VENDOR, CONTACT, BIDSTATUS, SCORE, ENTERDATE
FROM ingestion_framework_test.bid_data_exploration.rfqvendor
WHERE RFQNUM LIKE 'D-111808%'
ORDER BY ENTERDATE DESC

**Observations (updated after first real run):**
- `RFQVENDORID` is the per-invited-vendor primary key; `VENDOR` is a vendor **code**, not a name — a vendor/company master table is needed and isn't among our current 7 tables.
- `RFQNUM` values like `D1752-R1` suggest rounds might be modeled as distinct RFQ records rather than a column — needs reconciling with `rfq`'s `DISCOUNT_REVISION` theory.
- Several award/discount-total columns are typed `binary` and look encrypted/masked in the sample rather than genuinely populated — flagged as a potential blocker for reading real pricing/award totals directly from this table.